# Business Forecasting — Chapter 2: Time-Series Graphics
## STUDENT LIVE CLASS VERSION

This is the notebook displayed during class. Run the cells in order from top to bottom.

**Important:** Do not worry about memorizing Python syntax today. Focus on what each graph tells us about a time series and forecasting.


# 1. Setup: the Business Forecasting Python toolkit

**Learning goals**
- Understand what a Python library is.
- Learn the standard imports we will reuse throughout Business Forecasting.
- Learn why we use short names such as `pd`, `np`, `plt`, and `sns`.

**Key idea:** Python becomes useful for forecasting because libraries provide reliable data-handling, graphing, and statistical tools.


In [ ]:
# Business Forecasting STANDARD SETUP — you will use these libraries often in this course.

import pandas as pd                    # pandas: tables, CSV files, dates, filtering, grouping
import numpy as np                     # NumPy: numerical calculations and random numbers
import matplotlib.pyplot as plt        # Matplotlib: basic plotting
import seaborn as sns                  # Seaborn: convenient statistical graphics
import statsmodels.api as sm           # statsmodels: statistical/time-series tools
from statsmodels.graphics.tsaplots import plot_acf  # ACF graph used repeatedly in forecasting

sns.set_theme(style="whitegrid")       # Give our graphs a clean, readable background
pd.set_option("display.precision", 3)  # Show most decimal numbers to 3 places

print("Business Forecasting setup complete.")

# 2. DataFrames: how Python stores business data

A **DataFrame** is a two-dimensional table. Rows are observations; columns are variables. Before using a real dataset, we will build a tiny one ourselves.

**Key idea:** A DataFrame is the standard table structure we will use for most Business Forecasting work.


In [ ]:
# Create a Python dictionary. Each key will become a column name.
example_data = {
    "Year": [2019, 2020, 2021, 2022, 2023],
    "Sales": [120, 135, 128, 155, 171]
}

# Convert the dictionary into a pandas DataFrame (a table).
sales_df = pd.DataFrame(example_data)

# Display the whole small table.
sales_df

# 3. Inspecting a DataFrame before analysis

Never begin forecasting by blindly graphing or modelling. First inspect the structure, data types, missing values, and a few rows.

**Key idea:** Inspection comes before forecasting.


In [ ]:
print("Shape:", sales_df.shape)            # (number of rows, number of columns)
print("Columns:", sales_df.columns.tolist()) # Convert column labels to a normal Python list
print("\nData types:")
print(sales_df.dtypes)                     # Show the data type stored in each column

print("\nFirst rows:")
display(sales_df.head())                   # Show the first 5 rows

print("\nSummary:")
display(sales_df.describe())               # Numerical summary: count, mean, std, min, quartiles, max

# 4. Dates and time stamps

Forecasting data are ordered in time. Python must understand a date as a date, not merely as text. We will learn `pd.to_datetime()` and the `.dt` date accessor.

**Key idea:** Time-series analysis requires correctly parsed dates.


In [ ]:
# Start with date values written as text.
dates = pd.Series(["2024-01-01", "2024-02-01", "2024-03-01"])

# Convert the text into true pandas datetime values.
dates = pd.to_datetime(dates)

# Build a small monthly time-series DataFrame.
monthly_df = pd.DataFrame({
    "Month": dates,
    "Demand": [210, 225, 238]
})

# Extract useful pieces of the date using the .dt accessor.
monthly_df["Year"] = monthly_df["Month"].dt.year
monthly_df["MonthName"] = monthly_df["Month"].dt.month_name()

monthly_df

# 5. Loading real Chapter 2 data from a CSV

We now move to real data. `pd.read_csv()` reads a comma-separated file directly into a DataFrame. `parse_dates=` tells pandas which column should be interpreted as a date while loading.

**Key idea:** Load data carefully and inspect it immediately.


In [ ]:
# URL of the Pharmaceutical Benefits Scheme dataset used in Chapter 2.
pbs_url = "https://otexts.com/fpppy/data/PBS_unparsed.csv"

# Read the CSV file directly from the book website.
# parse_dates=["Month"] tells pandas to convert Month to datetime immediately.
pbs = pd.read_csv(pbs_url, parse_dates=["Month"])

# Keep the columns we need for this lesson.
pbs = pbs[["Month", "Concession", "Type", "ATC1", "ATC2", "Scripts", "Cost"]]

# Check the result before doing analysis.
print(pbs.info())
display(pbs.head())

# 6. Filtering rows with `.loc[]`

Filtering means keeping only observations that meet a condition. Here we select pharmaceutical records whose ATC2 code is `A10`.

**Key idea:** Filtering turns a large dataset into the specific business series you want to study.


In [ ]:
# Create a Boolean condition: True where ATC2 equals "A10", False elsewhere.
is_a10 = pbs["ATC2"] == "A10"

# .loc[condition] keeps only rows where the condition is True.
a10 = pbs.loc[is_a10].copy()

# Display a sample of the filtered data.
display(a10.head())

# Check how many rows remain.
print("Rows after filtering:", len(a10))

# 7. Aggregation with `groupby()`

The A10 data contain several records per month because of different concession and payment categories. To obtain one monthly cost series, we group by month and sum Cost.

**Key idea:** `groupby()` is how we turn detailed transactional records into a usable time series.


In [ ]:
# Group all A10 observations that belong to the same Month.
# as_index=False keeps Month as a normal column.
total_cost_df = (
    a10.groupby("Month", as_index=False)
       .agg({"Cost": "sum"})
)

# Convert dollars to millions of dollars for easier graph labels.
total_cost_df["CostMillions"] = total_cost_df["Cost"] / 1_000_000

# Keep only the two columns needed for the time series.
total_cost_df = total_cost_df[["Month", "CostMillions"]]

display(total_cost_df.head())

# 8. Seasonal period: the number you must identify

The **seasonal period** is the number of observations required for a seasonal pattern to repeat.

Common examples:
- quarterly data → 4 observations per year;
- monthly data → 12;
- weekly data → about 52;
- daily data with weekly seasonality → 7;
- hourly data with daily seasonality → 24.

**Key idea:** Frequency tells how often we observe; seasonal period tells how often a repeating pattern returns.


In [ ]:
seasonal_period_examples = pd.DataFrame({
    "Data frequency": ["Quarterly", "Monthly", "Weekly", "Daily", "Hourly"],
    "Common seasonal period": [4, 12, 52, 7, 24],
    "Pattern represented": ["Year", "Year", "Year", "Week", "Day"]
})

seasonal_period_examples

# 9. Time plots: the first graph in forecasting

A **time plot** places time on the horizontal axis and the observed value on the vertical axis. Consecutive observations are connected so we can see structure over time.

**Key idea:** Graphs are diagnostic tools, not decoration.


In [ ]:
# Create a plotting area.
fig, ax = plt.subplots(figsize=(10, 4))

# Draw CostMillions against Month.
ax.plot(total_cost_df["Month"], total_cost_df["CostMillions"])

# Add labels that explain exactly what the viewer is seeing.
ax.set(
    title="Australian antidiabetic drug sales",
    xlabel="Month",
    ylabel="Cost (millions of Australian dollars)"
)

# Display the graph.
plt.show()

# 10. Trend, seasonality, and cycles

Three terms must be distinguished:
- **Trend:** long-term movement in the level.
- **Seasonality:** a repeating pattern with a known, fixed calendar frequency.
- **Cycle:** rises and falls without a fixed frequency.

**Key idea:** Dates contain useful structure that can be extracted for analysis.


In [ ]:
# Add year and month information so we can examine the monthly seasonal pattern.
season_df = total_cost_df.copy()
season_df["Year"] = season_df["Month"].dt.year
season_df["MonthNumber"] = season_df["Month"].dt.month
season_df["MonthName"] = season_df["Month"].dt.strftime("%b")

# Preview the new columns.
display(season_df.head())

# 11. Seasonal plot

A seasonal plot overlays each year's values against the months of the year. This makes the within-year seasonal pattern much easier to see.

**Key idea:** A seasonal plot compares the same positions within each seasonal cycle.


In [ ]:
# Define the correct month order so the x-axis runs Jan to Dec.
month_order = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

# Create the graph.
fig, ax = plt.subplots(figsize=(10, 5))

# Draw one line for each year.
sns.lineplot(
    data=season_df,
    x="MonthName",
    y="CostMillions",
    hue="Year",
    sort=False,
    ax=ax
)

# Force calendar month order and add labels.
ax.set_xticks(range(12), month_order)
ax.set(
    title="Seasonal plot: antidiabetic drug sales",
    xlabel="Month of year",
    ylabel="Cost (millions)"
)

# Move the legend outside so it does not cover the data.
ax.legend(title="Year", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.show()

# 12. Seasonal subseries plot

A **seasonal subseries plot** groups all January observations together, all February observations together, and so on. It helps compare seasonal levels across time.

**Key idea:** Subseries plots separate seasons so their levels and changes can be compared directly.


In [ ]:
# Create a 3-by-4 grid: one small graph for each month.
fig, axes = plt.subplots(3, 4, figsize=(12, 8), sharey=True)
axes = axes.flatten()

# Loop through months 1 to 12.
for month_number in range(1, 13):
    ax = axes[month_number - 1]

    # Keep only observations belonging to the current month.
    one_month = season_df.loc[season_df["MonthNumber"] == month_number]

    # Plot that month's values across years.
    ax.plot(one_month["Year"], one_month["CostMillions"], marker="o")

    # Add a horizontal line showing the average for this month.
    ax.axhline(one_month["CostMillions"].mean(), linestyle="--")

    # Add the month name as the panel title.
    ax.set_title(month_order[month_number - 1])

fig.suptitle("Seasonal subseries plot: antidiabetic drug sales")
fig.supxlabel("Year")
fig.supylabel("Cost (millions)")
plt.tight_layout()
plt.show()

# 13. Multiple seasonal periods: electricity demand

Some data have more than one repeating seasonal pattern. Half-hourly electricity demand can have daily, weekly, and annual patterns.

**Key idea:** A time series can repeat at more than one frequency.


In [ ]:
# Load Victorian half-hourly electricity data from Chapter 2.
vic_url = "https://otexts.com/fpppy/data/vic_elec.csv"
vic_elec = pd.read_csv(vic_url, parse_dates=["ds"])

# Keep only the Demand series.
demand = vic_elec.loc[vic_elec["unique_id"] == "Demand"].copy()

# Use only one week to make the classroom graph easy to read.
one_week = demand.loc[
    (demand["ds"] >= "2014-01-06") &
    (demand["ds"] <  "2014-01-13")
].copy()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(one_week["ds"], one_week["y"])
ax.set(
    title="Victorian electricity demand — one week",
    xlabel="Date and time",
    ylabel="Demand"
)
plt.xticks(rotation=45)
plt.show()

# 14. Scatterplots: relationships between variables

Forecasting sometimes uses predictor variables. A scatterplot helps us see the relationship between two numerical variables.
Here we compare electricity demand with temperature.

**Key idea:** Scatterplots reveal relationships that a correlation number can hide.


In [ ]:
# Reshape the data so Demand and Temperature become separate columns.
elec_wide = (
    vic_elec.pivot(index="ds", columns="unique_id", values="y")
            .reset_index()
)

# Keep observations from 2014.
elec_2014 = elec_wide.loc[elec_wide["ds"].dt.year == 2014].copy()

# Draw Temperature on x and Demand on y.
fig, ax = plt.subplots(figsize=(7, 5))
sns.scatterplot(
    data=elec_2014,
    x="Temperature",
    y="Demand",
    s=12,
    ax=ax
)
ax.set(
    title="Electricity demand versus temperature",
    xlabel="Temperature",
    ylabel="Electricity demand"
)
plt.show()

# 15. Correlation — useful, but not enough

The correlation coefficient measures the strength and direction of a **linear** relationship. It ranges from -1 to +1.

**Key idea:** Correlation summarizes linear association only.


In [ ]:
# Keep only rows where both variables are present.
corr_data = elec_2014[["Temperature", "Demand"]].dropna()

# Compute the Pearson correlation coefficient.
r = corr_data["Temperature"].corr(corr_data["Demand"])

print(f"Correlation between Temperature and Demand: {r:.3f}")

# 16. Lag: comparing a series with its own past

A **lag** moves a time series backward by a specified number of observations. `shift(1)` creates the previous observation, `shift(4)` creates the observation four periods earlier, and so on.

**Key idea:** A lag is a previous value aligned with the current observation.


In [ ]:
# Load quarterly Australian beer production.
beer_url = "https://otexts.com/fpppy/data/aus_production.csv"
beer = pd.read_csv(beer_url, parse_dates=["ds"])

# Keep only dates from 2000 onward and only the columns we need.
beer = beer.loc[beer["ds"] >= "2000", ["ds", "Beer"]].copy()

# Rename Beer to y so the target has a short generic name.
beer = beer.rename(columns={"Beer": "y"})

# Create lagged versions of the target.
beer["lag_1"] = beer["y"].shift(1)  # one quarter earlier
beer["lag_4"] = beer["y"].shift(4)  # four quarters = one year earlier

display(beer.head(8))

# 17. Lag plots

A lag plot places the current value \(y_t\) against a lagged value \(y_{t-k}\). Repeating the plot for different lags can reveal seasonal dependence.

**Key idea:** Lag plots show how current values relate to their own past.


In [ ]:
# Create four lag plots for lags 1 through 4.
fig, axes = plt.subplots(2, 2, figsize=(8, 8))

for lag, ax in zip(range(1, 5), axes.flatten()):
    lagged = beer["y"].shift(lag)

    ax.scatter(lagged, beer["y"], s=20)
    ax.set(
        title=f"Lag {lag}",
        xlabel=f"y(t-{lag})",
        ylabel="y(t)"
    )

plt.tight_layout()
plt.show()

# 18. Autocorrelation and the ACF

**Autocorrelation** is correlation between a time series and its own lagged values. The **ACF** displays autocorrelation across many lags.

**Key idea:** The ACF summarizes dependence between present values and many past lags.


In [ ]:
# Calculate autocorrelations from lag 0 through lag 12.
acf_values = sm.tsa.acf(beer["y"].dropna(), nlags=12, fft=False)

# Put the values in a small table.
acf_table = pd.DataFrame({
    "Lag": range(len(acf_values)),
    "ACF": acf_values
})

display(acf_table)

# Plot the ACF.
fig, ax = plt.subplots(figsize=(9, 4))
plot_acf(
    beer["y"].dropna(),
    lags=16,
    zero=False,
    ax=ax
)
ax.set(
    title="ACF of quarterly beer production",
    xlabel="Lag (quarters)",
    ylabel="Autocorrelation"
)
plt.show()

# 19. Trend and seasonality in an ACF

A trended series often has large positive autocorrelations that decline slowly. A seasonal series often has larger ACF spikes at seasonal lags.

**Key idea:** ACF patterns reflect structure already present in the time series.


In [ ]:
# Plot the ACF of monthly drug sales, which contains trend and seasonality.
fig, ax = plt.subplots(figsize=(10, 4))
plot_acf(
    total_cost_df["CostMillions"].dropna(),
    lags=48,
    zero=False,
    ax=ax
)
ax.set(
    title="ACF of monthly antidiabetic drug sales",
    xlabel="Lag (months)",
    ylabel="Autocorrelation"
)
plt.show()

# 20. White noise: the forecasting benchmark

A **white-noise** series has no predictable autocorrelation structure. Its observations fluctuate randomly around a stable level with roughly constant variation.

**Key idea:** If forecast errors still contain structure, the model has probably missed information.


In [ ]:
# Use a fixed random seed so everyone in class gets the same random numbers.
rng = np.random.default_rng(seed=30)

# Generate 50 random observations from a standard normal distribution.
white_noise = pd.Series(rng.normal(loc=0, scale=1, size=50), name="y")

# Plot the random series.
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(range(1, 51), white_noise)
ax.set(
    title="Example of white noise",
    xlabel="Observation",
    ylabel="Value"
)
plt.show()

# Plot its autocorrelation function.
fig, ax = plt.subplots(figsize=(9, 3))
plot_acf(white_noise, lags=16, zero=False, ax=ax)
ax.set(
    title="ACF of white noise",
    xlabel="Lag",
    ylabel="Autocorrelation"
)
plt.show()


# Class Work / Assignment — Chapter 2: Time-Series Graphics

## Purpose
This is not a coding-speed exercise. Your goal is to show that you can **inspect a time series, create appropriate graphics, and explain what the graphics mean for forecasting**.

### Dataset
Use the `tute1.csv` dataset from the FPP Python book:

`https://otexts.com/fpppy/data/tute1.csv`

It contains quarterly `Sales`, `AdBudget`, and `GDP` series.

## Part A — Data inspection
1. Import the dataset with `pd.read_csv()` and parse `ds` as a date.
2. Display the first 10 rows.
3. Use `.info()` and answer:
   - How many observations are present?
   - Which columns are numerical?
   - Is `ds` recognized as a date?
4. Identify the data frequency.
5. State the annual seasonal period for quarterly data.

## Part B — Time plots
6. Make a time plot of `Sales`.
7. Make a time plot of `AdBudget`.
8. Make a time plot of `GDP`.
9. For each graph, write 2–4 sentences discussing:
   - trend;
   - possible seasonality;
   - cycles;
   - unusual observations.

## Part C — Relationships
10. Make a scatterplot of `Sales` versus `AdBudget`.
11. Calculate their correlation.
12. Explain why the scatterplot should be examined **before** interpreting the correlation number.
13. Make a scatterplot of `Sales` versus `GDP`.
14. Which variable appears more useful as a possible predictor of Sales? Explain from the graphs, not only the correlation.

## Part D — Lag and ACF
15. Create `Sales_lag1` using `.shift(1)`.
16. Create `Sales_lag4` using `.shift(4)`.
17. Explain in words what lag 4 means for quarterly data.
18. Create a lag-1 scatterplot for Sales.
19. Create a lag-4 scatterplot for Sales.
20. Plot the ACF of Sales for at least 16 lags.
21. Is there evidence of seasonal dependence? Explain which lags support your answer.

## Part E — Forecasting interpretation
22. In one paragraph, describe the main time-series features of Sales.
23. What characteristics would a future forecasting model need to capture?
24. Why would fitting a model **without first plotting the series** be risky?
25. Write one question about the data that the graphs cannot answer by themselves.

### Submission
Submit your completed Colab notebook. Every graph must have a meaningful title and axis labels. Put your written answers in **Text/Markdown cells**, not as Python comments.

### Business Forecasting expectation
You are not graded only on whether Python runs. You are graded on whether you can explain **what the output means for forecasting**.


In [ ]:
# ASSIGNMENT STARTER CELL
# Complete the questions above in additional Code and Text cells.

tute1_url = "https://otexts.com/fpppy/data/tute1.csv"
tute1 = pd.read_csv(tute1_url, parse_dates=["ds"])

display(tute1.head(10))
tute1.info()